# CSE422 Mushroom Classification Project — Full Dataset (Run this in Colab)

**Important:** This notebook is configured to train models on the full dataset (`mushroom_dataset.xlsx`, 61,069 rows). Run it in Google Colab (Runtime → Change runtime type → GPU recommended) if available. Training Random Forest on the full dataset can be slow; the notebook trains with 100 estimators by default — increase if you want better results.

Files saved by this notebook will appear in the working directory under `./{OUTPUT_DIRNAME}/`.

## 0. Upload dataset
If you're in Colab, upload `mushroom_dataset.xlsx` to the VM (Files sidebar → Upload) or mount Google Drive and place the dataset there.

In [ ]:
# If running in Colab, you can mount drive (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/path/to/mushroom_dataset.xlsx'  # update as needed

# If you uploaded directly to the Colab VM, place dataset in the working dir and use:
DATA_PATH = 'mushroom_dataset.xlsx'

## 1. Imports and helpers

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import joblib, json, os, zipfile
print('Libraries loaded')

## 2. Load dataset and quick EDA

In [ ]:
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
print('Dataset shape:', df.shape)
print(df.head())
print('\nMissing values per column:\n', df.isna().sum())
print('\nTarget distribution:\n', df['class'].value_counts())

## 3. Preprocessing & Pipelines (Full dataset)

In [ ]:
# Map target
df['class'] = df['class'].map({'p':1,'e':0})

X = df.drop(columns=['class'])
y = df['class']

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print('Numeric cols:', numeric_cols)
print('Categorical cols (sample):', categorical_cols[:10])

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=True))])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_cols),
                                              ('cat', categorical_transformer, categorical_cols)],
                                 remainder='drop')

## 4. Train / Evaluate on full dataset

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Train/test sizes:', X_train.shape, X_test.shape)

# Pipelines
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('clf', LogisticRegression(max_iter=1000, random_state=42))])
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))])

# Create outputs dir
OUT_DIR = Path('mushroom_project_outputs_full')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fit Logistic Regression
print('\nFitting Logistic Regression...')
lr_pipeline.fit(X_train, y_train)
joblib.dump(lr_pipeline, OUT_DIR / 'logistic_pipeline_full.joblib')
print('Saved logistic_pipeline_full.joblib')

# Fit Random Forest (this may take several minutes)
print('\nFitting Random Forest (n_estimators=100) ...')
rf_pipeline.fit(X_train, y_train)
joblib.dump(rf_pipeline, OUT_DIR / 'rf_pipeline_full.joblib')
print('Saved rf_pipeline_full.joblib')

# Predictions
lr_pred = lr_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:,1]
rf_proba = rf_pipeline.predict_proba(X_test)[:,1]

# Metrics and save reports
lr_report = classification_report(y_test, lr_pred, output_dict=True)
rf_report = classification_report(y_test, rf_pred, output_dict=True)
with open(OUT_DIR / 'lr_classification_report.json', 'w') as f:
    json.dump(lr_report, f, indent=2)
with open(OUT_DIR / 'rf_classification_report.json', 'w') as f:
    json.dump(rf_report, f, indent=2)

print('\nLogistic Regression report:\n', classification_report(y_test, lr_pred))
print('\nRandom Forest report:\n', classification_report(y_test, rf_pred))

# AUC
print('\nLR AUC:', roc_auc_score(y_test, lr_proba))
print('RF AUC:', roc_auc_score(y_test, rf_proba))

# Confusion matrices & ROC curves
cm_lr = confusion_matrix(y_test, lr_pred)
cm_rf = confusion_matrix(y_test, rf_pred)

import matplotlib.pyplot as plt
def save_confusion(cm, title, fname):
    plt.figure(figsize=(4,4))
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.colorbar()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i,j]), ha='center', va='center')
    plt.tight_layout()
    plt.savefig(fname)
    plt.close()

save_confusion(cm_lr, 'LR Confusion (full)', OUT_DIR / 'confusion_lr_full.png')
save_confusion(cm_rf, 'RF Confusion (full)', OUT_DIR / 'confusion_rf_full.png')

# ROC
from sklearn.metrics import roc_curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
plt.figure()
plt.plot(fpr_lr, tpr_lr)
plt.plot(fpr_rf, tpr_rf)
plt.plot([0,1],[0,1])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (full)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'roc_curves_full.png')
plt.close()

print('\nAll artifacts saved to', OUT_DIR)

## 5. Package outputs
After running all cells, you can download a zip of the outputs using the cell below.

In [ ]:
import zipfile, os
OUT_DIR = 'mushroom_project_outputs_full'
ZIP_PATH = 'mushroom_project_outputs_full.zip'
with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
    for root, dirs, files in os.walk(OUT_DIR):
        for fname in files:
            full = os.path.join(root, fname)
            arc = os.path.relpath(full, OUT_DIR)
            zf.write(full, arcname=arc)
print('Zipped outputs to', ZIP_PATH)